# Lesson 2.2: A conversation, a dispatcher and a visible result

Read the follow-up in the first block against the two earlier messages. Then find the separate operation that looks up the order. The program checks the operation name, order identifier and caller's permitted orders before it retrieves anything. The last block shows how successive pieces of text build the display.

[Companion lesson](../modules/02-conversation/02-code-workshop.md) · [Original-source walkthrough](../modules/02-conversation/05-source-walkthrough.md)

Run one cell at a time from top to bottom. The programme uses the fictional records shown in the code. No API key or model download is needed.

## Predict the result

Read `messages` first, then `result`, then the final element of `updates`. Which one contains the order lookup result?

In [ ]:
history = [
    {"role": "user", "content": "Check order O-17"},
    {"role": "assistant", "content": "I will retrieve the order record."},
]
current_message = {"role": "user", "content": "What quantities are recorded?"}
messages = history + [current_message]

orders = {"O-17": {"ordered": 12, "reported_received": 10}}

def lookup_order(order_id):
    record = orders.get(order_id)
    if record is None:
        return {"status": "not_found", "order_id": order_id}
    return {"status": "found", "order_id": order_id, "record": record}

def dispatch(tool_name, arguments, permitted_order_ids):
    tools = {"lookup_order": lookup_order}
    if tool_name not in tools:
        raise ValueError("Unknown tool")
    if set(arguments) != {"order_id"}:
        raise ValueError("Expected only order_id")
    order_id = arguments["order_id"]
    if not isinstance(order_id, str):
        raise ValueError("order_id must be text")
    if order_id not in permitted_order_ids:
        raise PermissionError("Order is outside the permitted scope")
    return tools[tool_name](order_id)

request = {"name": "lookup_order", "arguments": {"order_id": "O-17"}}
result = dispatch(request["name"], request["arguments"], {"O-17"})

def render_in_parts(parts):
    current = ""
    for part in parts:
        current = current + part
        yield current

updates = list(render_in_parts(["Order O-17: ", "12 ordered; ", "10 reported received."]))

## Inspect the result

`print` displays values below the cell. When present, `assert` checks a specific expectation and raises an error if it fails.

In [ ]:
print(messages, result, updates[-1], sep="\n")
assert result["record"]["reported_received"] == 10

## Try a change

Ask for O-17 again, but grant access only to O-18. Which check should stop the lookup?

Make a prediction before opening the worked answer.

In [ ]:
# Try the change here, or make a temporary edit in the first code cell.

<details>
<summary>Worked answer</summary>

The order is outside the permitted set. The earlier conversation did not confer access.

```python
try:
    dispatch("lookup_order", {"order_id": "O-17"}, {"O-18"})
except PermissionError as error:
    print(error)
else:
    raise AssertionError("The lookup should be denied")
```

</details>